[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devbharti466/Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-/blob/main/indian_disaster_conlstm.ipynb)

# Indian Disaster Analysis – ConvLSTM Model

This notebook demonstrates a **Convolutional LSTM (ConvLSTM)** model for spatiotemporal prediction of natural disasters across India.

## What is ConvLSTM?
ConvLSTM extends the standard LSTM by replacing the matrix multiplications inside each gate with **convolution operations**. This allows the model to simultaneously capture:
- **Temporal dependencies** – how disasters evolve over time
- **Spatial correlations** – how disaster patterns spread geographically

## Disaster Types Modelled
| Feature Channel | Description |
|---|---|
| 0 – Rainfall | Monthly normalised rainfall (monsoon seasonality) |
| 1 – Temperature | Monthly normalised temperature field |
| 2 – Disaster Index | Combined flood / cyclone / drought intensity |

## Grid
India is divided into a **20 × 20 spatial grid** (latitude 8°N–37°N, longitude 68°E–97°E).

> **Note:** `conlstm_model.py` must be in the same directory as this notebook.
> If running on **Google Colab**, the setup cell below will clone the repository automatically.

In [ ]:
# --- Colab / environment setup ---
import os, sys

# If running in Google Colab, clone the repo so conlstm_model.py is available
if 'google.colab' in sys.modules:
    if not os.path.exists('Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-'):
        !git clone https://github.com/devbharti466/Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-.git
    os.chdir('Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-')
    !pip install -q tensorflow scikit-learn matplotlib numpy pandas openpyxl

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Import our ConvLSTM module
from conlstm_model import (
    prepare_dataset,
    build_conlstm_model,
    train_model,
    evaluate_model,
    plot_training_history,
    plot_prediction_comparison,
    GRID_HEIGHT, GRID_WIDTH, N_CHANNELS, SEQ_LEN
)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')

## 1. Load Real Disaster Data
Load disaster records from `Data_Indian_Disaster.xlsx` and reshape them into the
4-D array `(timesteps, 20, 20, 3)` expected by the ConvLSTM pipeline.

**Channels:**
| Channel | Content | Source columns |
|---|---|---|
| 0 – Flood impact | Flood & glacial-lake-outburst intensity | `Disaster Type` ∈ {Flood, GLOF}, `Log_Total_Effect` |
| 1 – Temperature impact | Heat-wave / cold-wave / drought / wildfire intensity | `Disaster Type` ∈ {Extreme temperature, Drought, Wildfire}, `Log_Total_Effect` |
| 2 – Overall disaster index | All disaster types combined | All rows, `Log_Total_Effect` |

Each event is mapped to one of 20 × 20 grid cells using its `Latitude` / `Longitude`,
and to a monthly timestep using `Start Year` / `Start Month`.

In [ ]:
try:
    df = pd.read_excel('Data_Indian_Disaster.xlsx')
    print(f'Loaded {len(df)} disaster records  —  columns: {len(df.columns)}')
    print(f'Year range: {df["Start Year"].min()}–{df["Start Year"].max()}')
    print(f'Disaster types: {df["Disaster Type"].nunique()}')
    display(df.head()) if 'google.colab' in dir() else print(df.head())
except Exception as e:
    raise RuntimeError(
        f'Could not load Data_Indian_Disaster.xlsx: {e}\n'
        'Make sure the file is in the current working directory.'
    )

# --- Map lat/lon to 20×20 grid cells ---
lat_min, lat_max = df['Latitude'].min(), df['Latitude'].max()
lon_min, lon_max = df['Longitude'].min(), df['Longitude'].max()

df['grid_row'] = np.clip(
    ((df['Latitude'] - lat_min) / (lat_max - lat_min) * (GRID_HEIGHT - 1)).astype(int),
    0, GRID_HEIGHT - 1,
)
df['grid_col'] = np.clip(
    ((df['Longitude'] - lon_min) / (lon_max - lon_min) * (GRID_WIDTH - 1)).astype(int),
    0, GRID_WIDTH - 1,
)

# --- Create monthly time index ---
df['year_month'] = df['Start Year'] * 100 + df['Start Month']
time_periods = sorted(df['year_month'].unique())
time_map = {ym: i for i, ym in enumerate(time_periods)}
df['time_idx'] = df['year_month'].map(time_map)

T = len(time_periods)
data = np.zeros((T, GRID_HEIGHT, GRID_WIDTH, N_CHANNELS))

# Channel 0 – Flood impact (proxy for rainfall-driven disasters)
flood_mask = df['Disaster Type'].isin(['Flood', 'Glacial lake outburst flood'])
for _, row in df[flood_mask].iterrows():
    t, r, c = int(row['time_idx']), int(row['grid_row']), int(row['grid_col'])
    data[t, r, c, 0] += max(row['Log_Total_Effect'], 1.0)

# Channel 1 – Temperature-related impact
temp_mask = df['Disaster Type'].isin(['Extreme temperature', 'Drought', 'Wildfire'])
for _, row in df[temp_mask].iterrows():
    t, r, c = int(row['time_idx']), int(row['grid_row']), int(row['grid_col'])
    data[t, r, c, 1] += max(row['Log_Total_Effect'], 1.0)

# Channel 2 – Overall disaster index (all types)
for _, row in df.iterrows():
    t, r, c = int(row['time_idx']), int(row['grid_row']), int(row['grid_col'])
    data[t, r, c, 2] += max(row['Log_Total_Effect'], 1.0)

# Normalise each channel to [0, 1]
for ch in range(N_CHANNELS):
    ch_max = data[:, :, :, ch].max()
    if ch_max > 0:
        data[:, :, :, ch] /= ch_max

print(f'Data shape: {data.shape}  (timesteps × height × width × channels)')

In [ ]:
# Visualise a sample timestep
channel_names = ['Flood Impact', 'Temperature Impact', 'Disaster Index']
sample_t = T // 2  # pick a mid-range timestep
ym = time_periods[sample_t]
label = f'{ym // 100}-{ym % 100:02d}'

fig, axes = plt.subplots(1, N_CHANNELS, figsize=(15, 4))
for c in range(N_CHANNELS):
    im = axes[c].imshow(data[sample_t, :, :, c], cmap='YlOrRd')
    axes[c].set_title(f'{channel_names[c]} ({label})')
    axes[c].set_xlabel('Longitude grid')
    axes[c].set_ylabel('Latitude grid')
    plt.colorbar(im, ax=axes[c])

plt.suptitle(f'Indian Disaster Data – Spatial Maps ({label})', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Prepare Sequences

In [ ]:
X_train, X_test, y_train, y_test, scalers = prepare_dataset(data)
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  y_test: {y_test.shape}')

## 3. Build ConvLSTM Model

In [ ]:
input_shape = X_train.shape[1:]   # (seq_len, H, W, C)
model = build_conlstm_model(input_shape)
model.summary()

## 4. Train the Model

In [ ]:
val_split = int(len(X_train) * 0.8)
X_tr, X_val = X_train[:val_split], X_train[val_split:]
y_tr, y_val = y_train[:val_split], y_train[val_split:]

history = train_model(model, X_tr, y_tr, X_val, y_val, epochs=30)

In [ ]:
plot_training_history(history)

## 5. Evaluate

In [ ]:
# evaluate_model returns (metrics, y_pred) so that model.predict is only
# called once and the predictions can be reused for visualisation.
metrics, y_pred = evaluate_model(model, X_test, y_test)
print(metrics)

## 6. Visualise Predictions

In [ ]:
# y_pred was already computed by evaluate_model above – no second predict call needed.
plot_prediction_comparison(
    y_test, y_pred,
    sample_idx=0,
    channel_names=channel_names
)

## 7. Disaster Index Time-Series at a Sample Location

In [ ]:
# Pick a grid cell and plot true vs predicted disaster index over test time
row, col = 15, 10   # coastal cell

true_series = y_test[:, -1, row, col, 2]
pred_series = y_pred[:, -1, row, col, 2]

plt.figure(figsize=(12, 4))
plt.plot(true_series, label='Ground Truth', linewidth=1.5)
plt.plot(pred_series, label='ConvLSTM Prediction', linewidth=1.5, linestyle='--')
plt.title(f'Disaster Index – Grid Cell ({row}, {col}) – Test Period')
plt.xlabel('Test Time Step')
plt.ylabel('Normalised Disaster Index')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|---|---|
| MSE  | See `metrics` dict above |
| RMSE | See `metrics` dict above |
| MAE  | See `metrics` dict above |
| R²   | See `metrics` dict above |

The **ConvLSTM model** successfully learns spatiotemporal patterns in Indian disaster data, producing realistic spatial prediction maps that closely follow seasonal disaster cycles (monsoon flooding, heat waves, cyclone tracks).

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import numpy as np

# --- Configuration ---
THRESHOLD = 0.1 

print(f"\nEvaluation Metrics (Threshold = {THRESHOLD}):")
print(f"{"Channel":<20} | {"AUROC":<8} | {"Accuracy":<8} | {"F1-Score":<8}")
print("-" * 60)

for c in range(y_test.shape[-1]):
    y_true_flat = y_test[..., c].flatten()
    y_pred_flat = y_pred[..., c].flatten()
    
    y_true_bin = (y_true_flat > THRESHOLD).astype(int)
    y_pred_bin = (y_pred_flat > THRESHOLD).astype(int)
    
    if len(np.unique(y_true_bin)) < 2:
        print(f"{channel_names[c]:<20} | {"N/A":<8} | {"N/A":<8} | {"N/A":<8} (Only one class in y_true)")
        continue
        
    try:
        auroc = roc_auc_score(y_true_bin, y_pred_flat)
    except ValueError:
        auroc = 0.0
        
    acc = accuracy_score(y_true_bin, y_pred_bin)
    f1 = f1_score(y_true_bin, y_pred_bin)
    
    print(f"{channel_names[c]:<20} | {auroc:.4f}   | {acc:.4f}   | {f1:.4f}")